# SHAP — interpretación del modelo ganador (Random Forest, v4: anillo 7–15 km, ratio 1:1)

## Configuración
- **Dataset:** `tuning/v4` — mismo `pixel_year_full.csv`, semilla 42, criterio de exclusión por año,
  pero con la geometría de muestreo de v4: pseudo-ausencias en un **anillo 7–15 km** alrededor de cada
  píxel quemado del mismo año (no el buffer simple de 3 km de v1/v3). Muestreo estratificado por año
  para replicar la distribución anual de las presencias, igual que en el notebook de v4.
- **Modelo:** Random Forest con los hiperparámetros de `tuning/v4` (se cargan de
  `v4_best_hyperparameters.csv` si existen; si no, fallback `n_estimators=500, max_features='sqrt',
  min_samples_leaf=5, max_depth=None`). Sin `class_weight='balanced'` forzado — el dataset v4 ya es 1:1
  por diseño, igual que en el `GridSearchCV` original de v4.
- **Explainer:** `shap.TreeExplainer` (exacto para árboles). Se explica la clase positiva (quemado).

### Por qué v4 y no v1/v3
v4 existe porque a 3 km el correlograma de Moran's I seguía alto (I≈0.41) y `dist_roads`/`dist_mosaic`/
`dist_coca` quedaban injustamente bajos: presencias y pseudo-ausencias compartían el mismo entorno
cercano y no había nada que separar. El anillo 7–15 km rompe esa dependencia espacial. Este notebook
comprueba si esas variables antropogénicas **resurgen** en SHAP una vez corregido el muestreo — la
segunda motivación de v4, explícitamente diferida en su §1.

## Por qué SHAP y no la importancia por impureza
La impureza está sesgada hacia variables continuas con muchos cortes (NDVI, clima) y penaliza
distribuciones concentradas (distancias con muchos ceros). SHAP reparte la contribución de forma
justa por predicción, sin ese sesgo.

## Qué hace este notebook
1. Reconstruye el dataset v4 (anillo 7–15 km, ratio 1:1). 2. Entrena el RF final con los hiperparámetros
de `tuning/v4`. 3. Calcula SHAP (beeswarm, barra, dependence plots). 4. Importancia por permutación
(verificación convergente). 5. Guarda tabla comparativa y figuras en `outputs/shap/SHAP_V2/`.


In [6]:
# === SHAP setup: dataset v4 (anillo 7-15 km, ratio 1:1) + Random Forest final (hiperparametros de tuning/v4) ===

import os, glob, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import shap

def find_file(fname):
    for p in ([f'../../data/model_dataset/{fname}']
              + glob.glob(f'../../**/{fname}', recursive=True)
              + glob.glob(f'../../../**/{fname}', recursive=True)):
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError(f"{fname} not found -- check the repo path")

PIXEL_YEAR_PATH = r"C:\Users\Natal\CEGE0049-Wildfire-Susceptibility-Assessment-in-Sabanas-de-Yar---Bajo-Cagu-n-Colombian-Amazon\data\model_dataset\pixel_year_full.csv"
pixel_year = pd.read_csv(PIXEL_YEAR_PATH if os.path.exists(PIXEL_YEAR_PATH) else find_file('pixel_year_full.csv'))

pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']
BLOCK = 0.25
INNER_BUFFER = 7000            # v4: radio interno del anillo
OUTER_BUFFER = 15000           # v4: radio externo del anillo
inner_deg = INNER_BUFFER / 111000
outer_deg = OUTER_BUFFER / 111000
RATIO = 1                      # ratio ganador identificado en tuning/v3, mantenido en v4
SEED = 42

presences = pixel_year[pixel_year['burned'] == 1].copy()
absences_all = pixel_year[pixel_year['burned'] == 0].copy()

kept_absences = []
for y in sorted(pixel_year['year'].unique()):
    pres_y = presences[presences['year'] == y][['lon','lat']].values
    abs_y  = absences_all[absences_all['year'] == y]
    if len(pres_y) == 0:
        kept_absences.append(abs_y); continue
    tree = cKDTree(pres_y)
    dists, _ = tree.query(abs_y[['lon','lat']].values, k=1)
    in_ring = (dists >= inner_deg) & (dists <= outer_deg)
    kept_absences.append(abs_y[in_ring])
absences_ring = pd.concat(kept_absences, ignore_index=True)

n_needed = int(round(RATIO * len(presences)))
print(f"Anillo {INNER_BUFFER/1000:.0f}-{OUTER_BUFFER/1000:.0f} km | "
      f"elegibles: {len(absences_ring):,} | requeridas: {n_needed:,}")
if len(absences_ring) < n_needed:
    print(f"  ADVERTENCIA: pool insuficiente, faltan {n_needed - len(absences_ring):,} pseudo-ausencias.")

# Muestreo estratificado por año: replica la distribucion anual de las presencias (igual que v4)
absences_sample = (absences_ring.groupby('year', group_keys=False)
                   .apply(lambda g: g.sample(
                       n=min(len(g), int((presences['year'] == g.name).sum() * RATIO)),
                       random_state=SEED)))

model_df = pd.concat([presences, absences_sample], ignore_index=True) \
             .sample(frac=1, random_state=SEED).reset_index(drop=True)

n_pres = int(model_df['burned'].sum())
print(f"Dataset v4 (anillo 7-15 km, ratio {RATIO}:1) ->", model_df.shape, f"({n_pres} presencias)")

# --- Hiperparametros del RF ganador de tuning/v4 ---
RF_PARAMS = {'n_estimators': 500, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'max_depth': None}

def _parse_param(v):
    if pd.isna(v):
        return None

    # If it's already numeric
    if isinstance(v, (int, np.integer)):
        return int(v)

    if isinstance(v, (float, np.floating)):
        # Convert whole numbers like 5.0 -> 5
        if v.is_integer():
            return int(v)
        return float(v)

    s = str(v)

    if s == "None":
        return None

    # literal_eval handles numbers, lists, dicts, booleans...
    try:
        return ast.literal_eval(s)
    except Exception:
        return s

try:
    v4_hp = pd.read_csv(find_file('v4_best_hyperparameters.csv'))
    rf_row = v4_hp[v4_hp['model'] == 'Random Forest']
    if len(rf_row):
        # Only keep columns that are non-NaN for this row -- the CSV has one row per
        # model but shares columns across all three (LR's clf__C, XGB's learning_rate,
        # etc.), so other models' params show up as NaN here and must be dropped, not
        # passed through as None.
        RF_PARAMS = {c: _parse_param(rf_row.iloc[0][c]) for c in v4_hp.columns
                     if c != 'model' and pd.notna(rf_row.iloc[0][c])}
        print("Loaded RF best_params from tuning/v4:", RF_PARAMS)
    else:
        print("Random Forest row not found in v4_best_hyperparameters.csv -> fallback:", RF_PARAMS)
except Exception as e:
    print("Could not read v4 hyperparameters -> fallback:", RF_PARAMS, "|", type(e).__name__)

X_df = model_df[pred_cols].copy()
X = X_df.values
y = model_df['burned'].astype(int).values
# No class_weight override -- matches v4's GridSearchCV RF exactly (dataset is already 1:1)
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, **RF_PARAMS)
rf.fit(X, y)
print("Final RF (v4) trained on", X.shape[0], "rows.")

SHAP_DIR = Path(r"C:\Users\Natal\CEGE0049-Wildfire-Susceptibility-Assessment-in-Sabanas-de-Yar---Bajo-Cagu-n-Colombian-Amazon\outputs\shap\SHAP_V2")
SHAP_DIR.mkdir(parents=True, exist_ok=True)
print("SHAP_V2 folder ready:", SHAP_DIR)


Anillo 7-15 km | elegibles: 14,928 | requeridas: 2,077
Dataset v4 (anillo 7-15 km, ratio 1:1) -> (4154, 13) (2077 presencias)
Loaded RF best_params from tuning/v4: {'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 500}
Final RF (v4) trained on 4154 rows.
SHAP_V2 folder ready: C:\Users\Natal\CEGE0049-Wildfire-Susceptibility-Assessment-in-Sabanas-de-Yar---Bajo-Cagu-n-Colombian-Amazon\outputs\shap\SHAP_V2


In [ ]:
# === SHAP values + beeswarm summary ===
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X)
if isinstance(sv, list):
    shap_pos = sv[1]
elif getattr(sv, 'ndim', 2) == 3:
    shap_pos = sv[:, :, 1]
else:
    shap_pos = sv
print("SHAP matrix shape:", np.asarray(shap_pos).shape)

plt.figure()
shap.summary_plot(shap_pos, X_df, feature_names=pred_cols, show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / 'shap_beeswarm.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved:", SHAP_DIR / 'shap_beeswarm.png')
# Read: right (positive SHAP) = pushes toward BURN. Colour = feature value (red high, blue low).
# For distances, low value = CLOSE.


In [ ]:
# === SHAP bar (mean |SHAP|) + dependence plots ===
plt.figure()
shap.summary_plot(shap_pos, X_df, feature_names=pred_cols, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / 'shap_bar.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved:", SHAP_DIR / 'shap_bar.png')

for feat in ['dist_mosaic', 'dist_roads']:
    plt.figure()
    shap.dependence_plot(feat, shap_pos, X_df, interaction_index=None, show=False)
    plt.tight_layout()
    out_path = SHAP_DIR / f'shap_dependence_{feat}.png'
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.show()
    print("Saved:", out_path)
# A steep drop near distance 0 that flattens = THRESHOLD effect (non-linear), unusable by a linear model.
# In v4 this is the key check: did the wider annulus restore contrast in dist_roads / dist_mosaic?


In [ ]:
# === Permutation importance + comparison of the 3 importance methods ===
perm = permutation_importance(rf, X, y, scoring='average_precision',
                              n_repeats=20, random_state=SEED, n_jobs=-1)
importance = pd.DataFrame({
    'predictor': pred_cols,
    'shap_mean_abs': np.abs(shap_pos).mean(axis=0),
    'impurity':      rf.feature_importances_,
    'permutation':   perm.importances_mean,
    'permutation_std': perm.importances_std,
})
for col in ['shap_mean_abs', 'impurity', 'permutation']:
    importance[f'rank_{col}'] = importance[col].rank(ascending=False).astype(int)
importance = importance.sort_values('shap_mean_abs', ascending=False).reset_index(drop=True)
print(importance.round(4).to_string(index=False))


In [ ]:
# === Save outputs ===
importance.to_csv(SHAP_DIR / 'shap_importance_comparison.csv', index=False)
print("Guardado:", SHAP_DIR / 'shap_importance_comparison.csv', "+ figuras PNG en", SHAP_DIR)

for feat in ['dist_mosaic', 'dist_roads', 'dist_coca']:
    row = importance[importance['predictor'] == feat]
    if len(row):
        r = row.iloc[0]
        print(f"{feat} -> SHAP #{r['rank_shap_mean_abs']} | "
              f"impurity #{r['rank_impurity']} | permutation #{r['rank_permutation']}")
